# Check skill name completeness
Check completeness of the skills name in row 1700 and before because that was when the code was not corrected

to be safe, we will check all rows

In [6]:
import pandas as pd

course_df = pd.read_csv(
    "data/2025-2026_module_clean_with_prereq_skillsfuture.csv",
    usecols=["extracted_skills"],
    encoding="utf-8-sig",
)

taxo_df = pd.read_csv(
    "data/skills_taxo.csv",
    usecols=["parent_skill_title"],
    encoding="utf-8-sig",
)

taxonomy_titles = {
    str(value).strip()
    for value in taxo_df["parent_skill_title"].dropna()
    if str(value).strip()
}

unmatched_skills_df = pd.DataFrame(
    {
        "unmatched_skill": sorted(
            {
                skill
                for raw_value in course_df["extracted_skills"].dropna()
                for skill in [token.strip() for token in str(raw_value).split("|")]
                if skill and skill not in taxonomy_titles
            }
        )
    }
)

print(f"Unique unmatched skills: {len(unmatched_skills_df)}")
unmatched_skills_df


Unique unmatched skills: 26


,unmatched_skill
0,Coaching and Assessment Management
1,Connection and Measurement
2,Cultural
3,Drainage and Gas Systems Design
4,Electrical
5,Electrical Termination
6,Electronic and Control Engineering
7,Environmental Management System Policies
8,Ethics
9,Generative AI Innovation


In [7]:
taxo_titles = (
    pd.read_csv(
        "data/skills_taxo.csv",
        usecols=["parent_skill_title"],
        encoding="utf-8-sig",
    )["parent_skill_title"]
    .dropna()
    .astype(str)
    .str.strip()
)

taxo_titles = [title for title in taxo_titles if title]

def find_prefix_comma_matches(skill):
    prefix = f"{skill},"
    matches = [title for title in taxo_titles if title.startswith(prefix)]
    return " | ".join(matches) if matches else None

unmatched_skills_df["matched_parent_skill_title"] = unmatched_skills_df["unmatched_skill"].apply(
    find_prefix_comma_matches
)

unmatched_skills_df


,unmatched_skill,matched_parent_skill_title
0,Coaching and Assessment Management,None
1,Connection and Measurement,None
2,Cultural,"Cultural, Heritage and Socio-economic Sensitiv..."
3,Drainage and Gas Systems Design,None
4,Electrical,"Electrical, Electronic and Control Engineering"
5,Electrical Termination,"Electrical Termination, Connection and Measure..."
6,Electronic and Control Engineering,None
7,Environmental Management System Policies,"Environmental Management System Policies, Stan..."
8,Ethics,"Ethics, Values and Legislation"
9,Generative AI Innovation,"Generative AI Innovation, Research and Develop..."


In [ ]:
# Check if any matched parent skill titles contain " | "
(unmatched_skills_df["matched_parent_skill_title"].fillna("").str.contains(r" \| ")).any()


np.False_

for any unmatched skill in the data frame, 
    
    if they are found in the 2025-2026_module_clean_with_prereq_skillsfuture, then replace it 
    
    if there is a matched_parent_skill_title, if not then remove it

In [9]:
import pandas as pd

# course_df = pd.read_csv("data/2025-2026_module_clean_with_prereq_skillsfuture.csv")
# unmatched_skills_df already exists

replacement_map = (
    unmatched_skills_df.loc[
        unmatched_skills_df["matched_parent_skill_title"].notna()
        & unmatched_skills_df["matched_parent_skill_title"].astype(str).str.strip().ne(""),
        ["unmatched_skill", "matched_parent_skill_title"],
    ]
    .drop_duplicates(subset=["unmatched_skill"])
    .set_index("unmatched_skill")["matched_parent_skill_title"]
    .to_dict()
)

remove_set = set(
    unmatched_skills_df.loc[
        unmatched_skills_df["matched_parent_skill_title"].isna()
        | unmatched_skills_df["matched_parent_skill_title"].astype(str).str.strip().eq(""),
        "unmatched_skill",
    ]
)

def clean_extracted_skills(raw_text):
    if pd.isna(raw_text) or not str(raw_text).strip():
        return raw_text

    tokens = [token.strip() for token in str(raw_text).split("|") if token.strip()]
    cleaned = []

    for token in tokens:
        if token in replacement_map:
            cleaned.append(replacement_map[token])
        elif token in remove_set:
            continue
        else:
            cleaned.append(token)

    # remove duplicates while preserving order
    cleaned = list(dict.fromkeys(cleaned))
    return " | ".join(cleaned)

course_df["extracted_skills"] = course_df["extracted_skills"].apply(clean_extracted_skills)

